In [ ]:
import pandas as pd
import requests
from datetime import datetime, date, timedelta

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
load_date = datetime.now()

nickname = "iadorokhin"

end_date = load_date - timedelta(days=1)
start_date = end_date - timedelta(days=30)



response = requests.get(
    url,
    params={
        "format": "geojson",
        "starttime": start_date.strftime('%Y-%m-%dT%H:%M:%S'),
        "endtime": end_date.strftime('%Y-%m-%dT%H:%M:%S')
    }, timeout=60,
)

response.raise_for_status()

data = response.json()

df = pd.json_normalize(data['features'])
if len(df) > 0:
    # 3. Добавляем технические колонки
    df["update_at"] = load_date
    df["start_bisness_date"] = start_date
    df["end_bisness_date"] = end_date


    str_load = load_date.strftime("%Y-%m-%d_%H-%M-%S")
    str_start = start_date.strftime("%Y-%m-%d")
    str_end = end_date.strftime("%Y-%m-%d")
        
    filename = f"{nickname}_{str_load}_({str_start}_{str_end}).csv"

    df.to_csv(filename, sep=';', index=False)
    print(f"Файл {filename} успешно сохранен. Строк: {len(df)}")
else:
    print("Данных за указанный период не найдено.")

